# 3. A transient operator: the model proposes the solution

**What you get out of this notebook:** the traced iteration of the tutorial,
reproduced step by step — incumbent at 9.236, a first sample the validator
refuses, a repaired tour at 8.000 — and the same trace coming out of the C++ port.

*Transient* means the artifact is the solution itself: it lives for this run of
the search and nothing survives it. Every candidate costs one model call.

In [1]:
import _bootstrap

import tsp_transient as tsp

## The instance and what conditions the operator

Five cities. The prompt carries the schema, the incumbent and its score: numeric
conditioning, the base channel of the tutorial's lens.

In [2]:
print(tsp.Spec().render([0, 2, 3, 4, 1], tsp.tour_len([0, 2, 3, 4, 1]), []))

[context] TSP toy instance; minimize Euclidean closed-tour length.
[conditioning] R = permutation of [0, 1, 2, 3, 4] starting at 0; incumbent [0, 2, 3, 4, 1], score 9.236.
[instruction] Emit one lower-length tour if possible.
[format] Use the CANDIDATE envelope.


## The run

In [3]:
best, best_score = tsp.main()

incumbent [0, 2, 3, 4, 1]  length 9.236
  step 0: accepted 8.0
  step 1: rejected 9.236
  step 2: rejected 10.893
best [0, 1, 2, 3, 4]  length 8.000


Read the three steps. The first one accepted a tour of length 8.0 — and it did so
*after* a repair, because the first completion in the pool is the infeasible
`[0, 1, 4, 4, 2]`. The repair does not appear as its own line in the log: it
happens inside the step. The two later steps propose valid but worse tours and the
acceptance rule turns them down.

## The same loop in C++

`cpp/tsp_transient.cpp` reads the same completions and prints the same lines. Both
are compared against `expected/tsp_transient.txt`, so if they ever disagree, the
algorithm as written in the paper is under-specified. Build it with `make -C cpp`.

In [4]:
import os, subprocess

binary = os.path.join('cpp', 'bin', 'tsp_transient')
if os.path.exists(binary):
    cpp = subprocess.run([binary], capture_output=True, text=True).stdout
    python = open(os.path.join('expected', 'tsp_transient.txt')).read()
    print(cpp)
    print('identical to the Python run:', cpp == python)
else:
    print('not built — run: make -C cpp')

incumbent [0, 2, 3, 4, 1]  length 9.236
  step 0: accepted 8.0
  step 1: rejected 9.236
  step 2: rejected 10.893
best [0, 1, 2, 3, 4]  length 8.000

identical to the Python run: True


Next: [4. An amortized operator](04_bpp_amortized.ipynb).